# ML Models for Well Performance Prediction

This notebook demonstrates the ML models for:
1. Production forecasting
2. Sensor anomaly detection
3. Predictive maintenance

## Setup

In [ ]:
import sys
sys.path.insert(0, '../backend')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from app.ml.models import WellPerformanceModel, ProductionForecaster, AnomalyDetector, PredictiveMaintenance

print("✓ Imports successful")

## 1. Production Forecasting Example

In [ ]:
# Generate synthetic production data
dates = [datetime.utcnow() - timedelta(days=90-i) for i in range(90)]
# Declining production trend typical of oil wells
production = [1000 - 2*i + np.random.normal(0, 20) for i in range(90)]
production = [max(p, 100) for p in production]  # Ensure non-negative

# Train forecaster
forecaster = ProductionForecaster()
forecaster.fit(dates, production)

# Make prediction
forecast = forecaster.predict(days_ahead=30)

print(f"✓ Trained forecaster on {len(dates)} historical records")
print(f"\nForecast for next 5 days:")
for date, predicted_bbl in forecast[:5]:
    print(f"  {date.date()}: {predicted_bbl:.1f} BBL")

In [ ]:
# Visualize forecast
plt.figure(figsize=(12, 5))

# Historical data
plt.plot(dates, production, 'o-', label='Historical Production', alpha=0.7, linewidth=2)

# Forecast
forecast_dates = [d for d, _ in forecast]
forecast_values = [v for _, v in forecast]
plt.plot(forecast_dates, forecast_values, 's--', label='Forecast', alpha=0.7, linewidth=2, color='orange')

plt.xlabel('Date')
plt.ylabel('Production (BBL)')
plt.title('Oil Production: Historical & Forecast')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Visualization complete")

## 2. Anomaly Detection Example

In [ ]:
# Generate synthetic sensor data
n_readings = 200
sensor_data = pd.DataFrame({
    'timestamp': [datetime.utcnow() - timedelta(hours=200-i) for i in range(n_readings)],
    'temperature_c': np.random.normal(85, 5, n_readings),
    'pressure_psi': np.random.normal(2500, 100, n_readings),
    'flow_rate_bpd': np.random.normal(750, 50, n_readings),
    'vibration_mms': np.random.normal(2.5, 0.5, n_readings),
})

# Add some anomalies
sensor_data.loc[50, 'temperature_c'] = 120  # High temperature
sensor_data.loc[100, 'vibration_mms'] = 8.5  # High vibration
sensor_data.loc[150, 'pressure_psi'] = 1500  # Pressure drop

print(f"✓ Generated {len(sensor_data)} sensor readings with 3 injected anomalies")
print(f"\nSensor data summary:\n{sensor_data.describe()}")

In [ ]:
# Train anomaly detector
feature_cols = ['temperature_c', 'pressure_psi', 'flow_rate_bpd', 'vibration_mms']

# Use 80% for training
train_data = sensor_data.iloc[:160]
detector = AnomalyDetector(contamination=0.1)
detector.fit(train_data, feature_cols)

# Detect anomalies in all data
anomalies = []
for idx, row in sensor_data.iterrows():
    reading_dict = row[feature_cols].to_dict()
    is_anomaly, score = detector.predict(reading_dict)
    if is_anomaly or score > 0.5:
        anomalies.append({
            'index': idx,
            'temperature_c': row['temperature_c'],
            'pressure_psi': row['pressure_psi'],
            'vibration_mms': row['vibration_mms'],
            'is_anomaly': is_anomaly,
            'score': score
        })

print(f"✓ Trained detector on {len(train_data)} readings")
print(f"\nAnomalies detected: {len(anomalies)}")
print("\nTop anomalies:")
for anom in sorted(anomalies, key=lambda x: x['score'], reverse=True)[:5]:
    print(f"  Index {anom['index']}: Score={anom['score']:.3f}, "
          f"Temp={anom['temperature_c']:.1f}°C, "
          f"Vibration={anom['vibration_mms']:.2f} mm/s")

In [ ]:
# Visualize anomalies
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

metrics = {
    'temperature_c': 'Temperature (°C)',
    'pressure_psi': 'Pressure (PSI)',
    'vibration_mms': 'Vibration (mm/s)',
    'flow_rate_bpd': 'Flow Rate (BPD)'
}

anomaly_indices = [a['index'] for a in anomalies]

for idx, (ax, (col, label)) in enumerate(zip(axes.flat, metrics.items())):
    ax.plot(sensor_data.index, sensor_data[col], 'o-', alpha=0.6, label='Normal')
    if anomaly_indices:
        ax.scatter(anomaly_indices, sensor_data.loc[anomaly_indices, col], 
                  color='red', s=100, label='Anomaly', zorder=5)
    ax.set_xlabel('Reading Index')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()
print("✓ Visualization complete")

## 3. Predictive Maintenance Example

In [ ]:
# Generate realistic sensor degradation pattern
n_days = 180
maintenance_data = pd.DataFrame({
    'date': [datetime.utcnow() - timedelta(days=n_days-i) for i in range(n_days)],
    'temperature_c': 80 + np.cumsum(np.random.normal(0.1, 0.5, n_days)),  # Gradual increase
    'pressure_psi': 2500 + np.random.normal(0, 50, n_days),
    'vibration_mms': 1.5 + np.cumsum(np.random.normal(0.02, 0.1, n_days)),  # Gradual increase
})

# Calculate health scores over time
maintenance = PredictiveMaintenance()
health_scores = []
for i in range(30, len(maintenance_data)):  # Use 30-day windows
    window = maintenance_data.iloc[i-30:i]
    score = maintenance.calculate_health_score(window)
    health_scores.append(score)

print(f"✓ Calculated health scores for {len(maintenance_data)} days")
print(f"\nCurrent health: {health_scores[-1]:.1f}/100")

# Check maintenance status
current_health = health_scores[-1]
needs_maint, message = maintenance.needs_maintenance(current_health)
print(f"Status: {message}")

In [ ]:
# Visualize health degradation
plt.figure(figsize=(12, 6))

dates_for_health = maintenance_data['date'].iloc[30:len(maintenance_data)]

# Plot health score
plt.plot(dates_for_health, health_scores, 'o-', linewidth=2, markersize=4, label='Health Score')

# Add threshold lines
plt.axhline(y=100, color='green', linestyle='--', alpha=0.5, label='Healthy (100)')
plt.axhline(y=60, color='yellow', linestyle='--', alpha=0.5, label='Warning (60)')
plt.axhline(y=40, color='red', linestyle='--', alpha=0.5, label='Critical (40)')

# Color zones
plt.axhspan(80, 100, alpha=0.1, color='green')
plt.axhspan(60, 80, alpha=0.1, color='yellow')
plt.axhspan(40, 60, alpha=0.1, color='orange')
plt.axhspan(0, 40, alpha=0.1, color='red')

plt.xlabel('Date')
plt.ylabel('Health Score')
plt.title('Equipment Health Degradation Over Time')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Visualization complete")

## 4. Comprehensive Well Analysis

In [ ]:
# Prepare complete well data
well_data = {
    'well_id': 1,
    'production_history': [
        {'date': d.isoformat(), 'oil_bbl': p}
        for d, p in zip(dates, production)
    ],
    'sensor_readings': [
        {
            'timestamp': row['timestamp'].isoformat(),
            'temperature_c': row['temperature_c'],
            'pressure_psi': row['pressure_psi'],
            'flow_rate_bpd': row['flow_rate_bpd'],
            'vibration_mms': row['vibration_mms'],
        }
        for _, row in sensor_data.iterrows()
    ]
}

# Run comprehensive analysis
model = WellPerformanceModel()
analysis = model.analyze_well(well_data)

print("\n" + "="*60)
print("COMPREHENSIVE WELL ANALYSIS")
print("="*60)
print(f"\n1. HEALTH SCORE: {analysis['health_score']:.1f}/100")
print(f"   Status: {analysis['maintenance_status']}")
print(f"   Message: {analysis['maintenance_message']}")

print(f"\n2. PRODUCTION FORECAST (Next 7 Days):")
for forecast_item in analysis['forecast'][:7]:
    print(f"   {forecast_item['date']}: {forecast_item['predicted_oil_bbl']:.1f} BBL")

print(f"\n3. ANOMALIES DETECTED: {len(analysis['anomalies'])}")
for anomaly in analysis['anomalies'][:3]:
    print(f"   Severity: {anomaly['severity']:.1f}% - {anomaly['reading']}")

print("\n" + "="*60)

## 5. Model Performance Metrics

In [ ]:
print("\nML MODEL SUMMARY:")
print("\n1. PRODUCTION FORECASTER")
print(f"   - Status: {'Trained' if forecaster.is_fitted else 'Not trained'}")
print(f"   - Training samples: {len(production)}")
print(f"   - Historical avg production: {np.mean(production):.1f} BBL")
print(f"   - Forecast avg (next 30d): {np.mean([v for _, v in forecast]):.1f} BBL")
print(f"   - Trend: {'Declining' if np.mean(production[:45]) > np.mean(production[45:]) else 'Stable/Growing'}")

print("\n2. ANOMALY DETECTOR")
print(f"   - Status: {'Trained' if detector.is_fitted else 'Not trained'}")
print(f"   - Training samples: {len(train_data)}")
print(f"   - Total test samples: {len(sensor_data)}")
print(f"   - Anomalies found: {len(anomalies)}")
print(f"   - Anomaly rate: {len(anomalies)/len(sensor_data)*100:.1f}%")

print("\n3. PREDICTIVE MAINTENANCE")
print(f"   - Analysis period: {n_days} days")
print(f"   - Initial health: {health_scores[0]:.1f}/100")
print(f"   - Current health: {health_scores[-1]:.1f}/100")
print(f"   - Degradation rate: {(health_scores[0]-health_scores[-1])/n_days*30:.2f} points/30 days")
print(f"   - Days until critical (est): {(health_scores[-1]-40)/(health_scores[0]-health_scores[-1])*n_days:.0f} days")

print("\n" + "="*60)